# Uncertainty-Aware Differential Drive Model

This notebook derives the symbolic equations used by `wheel_odometry_parametric` to compute wheel-based relative motion and its covariance.

The derivation is based on the differential-drive parametric uncertainty model from Filho et al., *The Impact of Parametric Uncertainties on Mobile Robots Velocities and Pose Estimation*, and adapts it to the active implementation in this workspace.

The implementation target is the relative pose increment consumed by GTSAM:

$$
\delta = \begin{bmatrix} \delta x \\ \delta y \\ \delta \psi \end{bmatrix},
\qquad
\Sigma_{rel} = \operatorname{Cov}(\delta x, \delta y, \delta \psi).
$$

The notebook keeps the notation explicit so each symbolic expression can be compared directly with the C++ implementation.


In [1]:
import sympy as sp

sp.init_printing(use_latex=True)


## Symbol Definitions

The symbols below use implementation-oriented names while preserving the mapping to Filho et al. The code uses `b` for full wheel separation, so `b = 2l` in the paper's notation. The lateral center-of-mass offset `y_c` corresponds to the paper's vertical offset parameter $\Delta P_Y$. ROS provides wheel angular positions, not wheel angular velocities, so the wheel rates used by the model are derived from consecutive encoder position readings.


In [2]:
# Construction parameters and center-of-mass offset
r_L, r_R = sp.symbols("r_L r_R", positive=True, real=True)
b = sp.symbols("b", positive=True, real=True)      # full wheel separation; b = 2*l in Filho et al.
y_c = sp.symbols("y_c", real=True)                 # lateral COM offset; y_c = Delta P_Y

# Consecutive encoder angular position readings
phi_L_km1 = sp.Symbol(r"\phi_L^{k-1}", real=True)
phi_L_k = sp.Symbol(r"\phi_L^k", real=True)
phi_R_km1 = sp.Symbol(r"\phi_R^{k-1}", real=True)
phi_R_k = sp.Symbol(r"\phi_R^k", real=True)
dt = sp.symbols("dt", positive=True, real=True)

# Parameter standard deviations
sigma_yc, sigma_b = sp.symbols("sigma_yc sigma_b", nonnegative=True, real=True)
sigma_rL, sigma_rR = sp.symbols("sigma_rL sigma_rR", nonnegative=True, real=True)

# Encoder angular position reading standard deviations
sigma_phi_L, sigma_phi_R = sp.symbols("sigma_phi_L sigma_phi_R", nonnegative=True, real=True)

# Group symbols by role
params = sp.Matrix([y_c, b, r_L, r_R])
param_stddevs = sp.Matrix([
    sigma_yc,
    sigma_b,
    sigma_rL,
    sigma_rR,
])

readings = sp.Matrix([
    phi_L_km1,
    phi_L_k,
    phi_R_km1,
    phi_R_k,
])
reading_stddevs = sp.Matrix([
    sigma_phi_L,
    sigma_phi_L,
    sigma_phi_R,
    sigma_phi_R,
])

# Derived quantities used by the velocity model
dphi_L = phi_L_k - phi_L_km1
dphi_R = phi_R_k - phi_R_km1
wheel_rate_L = dphi_L / dt
wheel_rate_R = dphi_R / dt

# Combined uncertain input vector and diagonal covariance
uncertain_inputs = params.col_join(readings)
uncertain_stddevs = param_stddevs.col_join(reading_stddevs)
Q_inputs = sp.diag(*[s**2 for s in uncertain_stddevs])

readings, reading_stddevs, params, param_stddevs, uncertain_inputs, Q_inputs


⎛                                                          ⎡    2              ↪
⎜                                                          ⎢σ_yc    0      0   ↪
⎜                                                          ⎢                   ↪
⎜                                                          ⎢          2        ↪
⎜                                         ⎡     y_c     ⎤  ⎢  0    σ_b     0   ↪
⎜                                         ⎢             ⎥  ⎢                   ↪
⎜                                         ⎢      b      ⎥  ⎢                 2 ↪
⎜                                         ⎢             ⎥  ⎢  0     0    σ_rL  ↪
⎜⎡\phi_L__{k-1}⎤  ⎡σ_φ_L⎤  ⎡y_c⎤  ⎡σ_yc⎤  ⎢     r_L     ⎥  ⎢                   ↪
⎜⎢             ⎥  ⎢     ⎥  ⎢   ⎥  ⎢    ⎥  ⎢             ⎥  ⎢                   ↪
⎜⎢  \phi_L__k  ⎥  ⎢σ_φ_L⎥  ⎢ b ⎥  ⎢σ_b ⎥  ⎢     r_R     ⎥  ⎢  0     0      0   ↪
⎜⎢             ⎥, ⎢     ⎥, ⎢   ⎥, ⎢    ⎥, ⎢             ⎥, ⎢                   ↪
⎜⎢\phi_R__{k-1}⎥  ⎢σ_φ_R⎥  ⎢

## Differential-Drive Kinematic Equations

The deterministic model starts from consecutive encoder angular position readings. Wheel angular rates are obtained by finite differencing those readings over the sample interval `dt`. The resulting differential-drive velocity model matches the current `computeWheelOdometry(...)` implementation and corresponds to Filho et al. Eq. 16/17 after substituting the full wheel separation `b = 2l`.


In [3]:
# Encoder increments from raw angular position readings
dphi_L = phi_L_k - phi_L_km1
dphi_R = phi_R_k - phi_R_km1

# Wheel angular rates derived from encoder increments
wheel_rate_L = dphi_L / dt
wheel_rate_R = dphi_R / dt

# Tangential velocity difference between right and left wheels
wheel_difference = r_R * wheel_rate_R - r_L * wheel_rate_L

# Robot body-frame linear velocity and yaw rate
v = (r_R * wheel_rate_R + r_L * wheel_rate_L) / 2 + y_c * wheel_difference / b
psi_dot = wheel_difference / b

twist = sp.Matrix([v, psi_dot])
twist


⎡                                                                        ⎛  r_ ↪
⎢                                                                    y_c⋅⎜- ── ↪
⎢r_L⋅(\phi_L__k - \phi_L__{k-1})   r_R⋅(\phi_R__k - \phi_R__{k-1})       ⎝     ↪
⎢─────────────────────────────── + ─────────────────────────────── + ───────── ↪
⎢             2⋅dt                              2⋅dt                           ↪
⎢                                                                              ↪
⎢                                       r_L⋅(\phi_L__k - \phi_L__{k-1})   r_R⋅ ↪
⎢                                     - ─────────────────────────────── + ──── ↪
⎢                                                     dt                       ↪
⎢                                     ──────────────────────────────────────── ↪
⎣                                                                      b       ↪

↪ L⋅(\phi_L__k - \phi_L__{k-1})   r_R⋅(\phi_R__k - \phi_R__{k-1})⎞⎤
↪ ───────────────────────────── + ──────

## Relative Increment Jacobian

The downstream GTSAM graph does not consume the global wheel pose uncertainty directly. It consumes a relative pose measurement between two graph keys. For one wheel-odometry interval from `k-1` to `k`, that measurement is

$$
\delta = \begin{bmatrix} \delta x \\ \delta y \\ \delta \psi \end{bmatrix}.
$$

This is why the covariance target is

$$
Q_\delta = \operatorname{Cov}(\delta x, \delta y, \delta \psi),
$$

not merely the scalar uncertainties of accumulated global pose. A GTSAM `BetweenFactor<Pose2>` needs the uncertainty of the relative measurement it is given.

The relative increment is computed from the body-frame velocity over the interval using midpoint integration:

$$
\delta \psi = dt\,\dot{\psi}, \qquad
\delta s = dt\,v,
$$

$$
\delta x = \delta s\cos\left(\frac{\delta \psi}{2}\right), \qquad
\delta y = \delta s\sin\left(\frac{\delta \psi}{2}\right).
$$

The Jacobian below computes every first-order sensitivity of that relative increment with respect to the uncertain construction parameters and raw encoder readings:

$$
J_\delta = \frac{\partial(\delta x, \delta y, \delta \psi)}
{\partial(y_c, b, r_L, r_R, \phi_L^{k-1}, \phi_L^k, \phi_R^{k-1}, \phi_R^k)}.
$$

Each entry in this matrix has a concrete meaning. For example, $\partial \delta x / \partial r_L$ is the local sensitivity of the relative forward displacement to the left wheel radius, while $\partial \delta \psi / \partial \phi_R^k$ is the local sensitivity of the relative yaw increment to the current right encoder reading. Multiplying this Jacobian by the input covariance propagates all of those first-order effects into the relative-pose covariance used downstream.


In [4]:
# Relative pose increment over one wheel-odometry interval
delta_yaw = dt * psi_dot
delta_s = dt * v
midpoint_yaw = delta_yaw / 2

delta_x = delta_s * sp.cos(midpoint_yaw)
delta_y = delta_s * sp.sin(midpoint_yaw)

relative_increment = sp.Matrix([delta_x, delta_y, delta_yaw])
relative_increment


⎡   ⎛                                                                        ⎛ ↪
⎢   ⎜                                                                    y_c⋅⎜ ↪
⎢   ⎜r_L⋅(\phi_L__k - \phi_L__{k-1})   r_R⋅(\phi_R__k - \phi_R__{k-1})       ⎝ ↪
⎢dt⋅⎜─────────────────────────────── + ─────────────────────────────── + ───── ↪
⎢   ⎝             2⋅dt                              2⋅dt                       ↪
⎢                                                                              ↪
⎢   ⎛                                                                        ⎛ ↪
⎢   ⎜                                                                    y_c⋅⎜ ↪
⎢   ⎜r_L⋅(\phi_L__k - \phi_L__{k-1})   r_R⋅(\phi_R__k - \phi_R__{k-1})       ⎝ ↪
⎢dt⋅⎜─────────────────────────────── + ─────────────────────────────── + ───── ↪
⎢   ⎝             2⋅dt                              2⋅dt                       ↪
⎢                                                                              ↪
⎢                           

In [5]:
# First-order sensitivity of [delta_x, delta_y, delta_yaw]
# with respect to [params; readings].
J_delta = relative_increment.jacobian(uncertain_inputs)
J_delta


⎡                                                                            ⎛ ↪
⎢                                                                            ⎜ ↪
⎢   ⎛  r_L⋅(\phi_L__k - \phi_L__{k-1})   r_R⋅(\phi_R__k - \phi_R__{k-1})⎞    ⎜ ↪
⎢dt⋅⎜- ─────────────────────────────── + ───────────────────────────────⎟⋅cos⎜ ↪
⎢   ⎝                dt                                dt               ⎠    ⎝ ↪
⎢───────────────────────────────────────────────────────────────────────────── ↪
⎢                                                                          b   ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                            ⎛ ↪
⎢                                                                            ⎜ ↪
⎢   ⎛  r_L⋅(\phi_L__k - \phi_L__{k-1})   r_R⋅(\phi_R__k - \phi_R__{k-1})⎞    ⎜ ↪
⎢dt⋅⎜- ─────────────────────

In [6]:
# First-order covariance propagation for the GTSAM relative pose factor.
Sigma_rel = J_delta * Q_inputs * J_delta.T
Sigma_rel


⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           

## Optimization

`Sigma_rel` is symmetric, so an implementation only needs the six unique upper-triangular entries: `Sigma_xx`, `Sigma_xy`, `Sigma_xyaw`, `Sigma_yy`, `Sigma_yyaw`, and `Sigma_yawyaw`.

For efficient runtime computation, split CSE into two stages:

- **Static precomputes:** factors that do not contain encoder readings. These can be computed once after calibration and standard-deviation parameters are loaded.
- **Dynamic precomputes:** factors that contain the current or previous encoder readings. These must be recomputed for each valid joint-state interval.

The split is performed using the raw reading symbols `phi_L^{k-1}`, `phi_L^k`, `phi_R^{k-1}`, and `phi_R^k`. The static CSE pass ignores substitutions containing any of those reading symbols; the dynamic CSE pass then reduces the remaining expressions.

The compact covariance is first rewritten back in terms of raw encoder readings using `Delta phi_L = phi_L^k - phi_L^{k-1}` and `Delta phi_R = phi_R^k - phi_R^{k-1}`. This keeps the cancellation of `dt` explicit while preserving the reading-level dependency boundary needed by the generated C++.

In [7]:
# Compact per-interval encoder increments.
Delta_phi_L, Delta_phi_R = sp.symbols(
    r"\Delta\phi_L \Delta\phi_R",
    real=True,
)

# Time-invariant factors are calibration constants and uncertainty settings.
time_invariant_factors = sp.Matrix([
    y_c,
    b,
    r_L,
    r_R,
    sigma_yc,
    sigma_b,
    sigma_rL,
    sigma_rR,
    sigma_phi_L,
    sigma_phi_R,
])

# Time-varying factors are the current encoder increments.
time_varying_factors = sp.Matrix([
    Delta_phi_L,
    Delta_phi_R,
])

# Rebuild the relative increment directly from encoder increments.
wheel_difference_inc = r_R * Delta_phi_R - r_L * Delta_phi_L
wheel_sum_inc = r_R * Delta_phi_R + r_L * Delta_phi_L

delta_yaw_compact = wheel_difference_inc / b
delta_s_compact = wheel_sum_inc / 2 + y_c * wheel_difference_inc / b

delta_x_compact = delta_s_compact * sp.cos(delta_yaw_compact / 2)
delta_y_compact = delta_s_compact * sp.sin(delta_yaw_compact / 2)

relative_increment_compact = sp.Matrix([
    delta_x_compact,
    delta_y_compact,
    delta_yaw_compact,
])
relative_increment_compact


⎡⎛\Delta\phi_L⋅r_L   \Delta\phi_R⋅r_R   y_c⋅(-\Delta\phi_L⋅r_L + \Delta\phi_R⋅ ↪
⎢⎜──────────────── + ──────────────── + ────────────────────────────────────── ↪
⎢⎝       2                  2                               b                  ↪
⎢                                                                              ↪
⎢⎛\Delta\phi_L⋅r_L   \Delta\phi_R⋅r_R   y_c⋅(-\Delta\phi_L⋅r_L + \Delta\phi_R⋅ ↪
⎢⎜──────────────── + ──────────────── + ────────────────────────────────────── ↪
⎢⎝       2                  2                               b                  ↪
⎢                                                                              ↪
⎢                                            -\Delta\phi_L⋅r_L + \Delta\phi_R⋅ ↪
⎢                                            ───────────────────────────────── ↪
⎣                                                             b                ↪

↪ r_R)⎞    ⎛-\Delta\phi_L⋅r_L + \Delta\phi_R⋅r_R⎞⎤
↪ ────⎟⋅cos⎜────────────────────────────────────⎟⎥
↪     

In [8]:
# Propagate covariance using the compact uncertain vector.
compact_uncertain_inputs = sp.Matrix([
    y_c,
    b,
    r_L,
    r_R,
    Delta_phi_L,
    Delta_phi_R,
])

Q_compact_inputs = sp.diag(
    sigma_yc**2,
    sigma_b**2,
    sigma_rL**2,
    sigma_rR**2,
    2 * sigma_phi_L**2,
    2 * sigma_phi_R**2,
)

J_delta_compact = relative_increment_compact.jacobian(compact_uncertain_inputs)
Sigma_rel_compact = J_delta_compact * Q_compact_inputs * J_delta_compact.T

# The compact relative-increment covariance no longer depends explicitly on dt.
dt in Sigma_rel_compact.free_symbols


False

In [9]:
# Six unique entries of the symmetric 3x3 covariance matrix.
# Use the compact no-dt form, then rewrite increments back to raw readings.
Sigma_rel_unique = sp.Matrix([
    Sigma_rel_compact[0, 0],
    Sigma_rel_compact[0, 1],
    Sigma_rel_compact[0, 2],
    Sigma_rel_compact[1, 1],
    Sigma_rel_compact[1, 2],
    Sigma_rel_compact[2, 2],
]).xreplace({
    Delta_phi_L: phi_L_k - phi_L_km1,
    Delta_phi_R: phi_R_k - phi_R_km1,
})

Sigma_rel_unique_names = sp.Matrix([
    sp.Symbol("Sigma_xx"),
    sp.Symbol("Sigma_xy"),
    sp.Symbol("Sigma_xyaw"),
    sp.Symbol("Sigma_yy"),
    sp.Symbol("Sigma_yyaw"),
    sp.Symbol("Sigma_yawyaw"),
])

dynamic_reading_symbols = {
    phi_L_km1,
    phi_L_k,
    phi_R_km1,
    phi_R_k,
}

Sigma_rel_static_replacements, Sigma_rel_after_static_cse = sp.cse(
    list(Sigma_rel_unique),
    symbols=sp.numbered_symbols("q_static_"),
    order="none",
    ignore=dynamic_reading_symbols,
)

Sigma_rel_dynamic_replacements, Sigma_rel_unique_optimized = sp.cse(
    Sigma_rel_after_static_cse,
    symbols=sp.numbered_symbols("q_"),
    order="none",
)

static_replacements_are_reading_free = all(
    not (expr.free_symbols & dynamic_reading_symbols)
    for _, expr in Sigma_rel_static_replacements
)

(
    len(Sigma_rel_static_replacements),
    len(Sigma_rel_dynamic_replacements),
    static_replacements_are_reading_free,
    Sigma_rel_static_replacements,
    Sigma_rel_dynamic_replacements,
    sp.Matrix.hstack(Sigma_rel_unique_names, sp.Matrix(Sigma_rel_unique_optimized)),
)


(15,
 34,
 True,
 [(q_static_0, sigma_b**2),
  (q_static_1, 1/b),
  (q_static_2, b**(-2)),
  (q_static_3, sigma_rL**2),
  (q_static_4, q_static_1*y_c),
  (q_static_5, sigma_rR**2),
  (q_static_6, r_L/2),
  (q_static_7, -q_static_4*r_L + q_static_6),
  (q_static_8, 2*sigma_phi_L**2),
  (q_static_9, r_R/2),
  (q_static_10, q_static_4*r_R + q_static_9),
  (q_static_11, sigma_phi_R**2),
  (q_static_12, 2*q_static_11),
  (q_static_13, sigma_yc**2),
  (q_static_14, q_static_1*r_L)],
 [(q_0, \phi_L^k - \phi_L^{k-1}),
  (q_1, q_0*r_L),
  (q_2, \phi_R^k - \phi_R^{k-1}),
  (q_3, -q_1 + q_2*r_R),
  (q_4, q_3*q_static_1),
  (q_5, q_4/2),
  (q_6, cos(q_5)),
  (q_7, q_3*q_static_2),
  (q_8, q_7*y_c),
  (q_9, q_1/2 + q_2*r_R/2 + q_4*y_c),
  (q_10, sin(q_5)),
  (q_11, q_10*q_3*q_9*q_static_2/2 - q_6*q_8),
  (q_12, q_9*q_static_1),
  (q_13, q_10*q_12),
  (q_14, -q_13*q_static_9 + q_6*q_static_10),
  (q_15, \phi_L^k/2 - \phi_L^{k-1}/2 - q_0*q_static_4),
  (q_16, q_13/2),
  (q_17, q_0*q_16 + q_15*q_6),
 

## Generate Factors

This cell writes the generated C++ precompute expressions only when `GENERATE_FACTORS` is set to `True`. It is disabled by default so re-running the notebook does not repeatedly overwrite the generated `.inl` files.

The output is split into static and dynamic files. Static precomputes are reading-free and can be evaluated once after parameters are loaded. Dynamic precomputes depend on encoder readings and must be evaluated for each valid joint-state interval.

In [10]:
from pathlib import Path

# Keep this disabled for normal notebook runs.
GENERATE_FACTORS = False

if GENERATE_FACTORS:
    cwd = Path.cwd()
    workspace_root = cwd if (cwd / "src" / "bot_controller").exists() else cwd.parent
    generated_dir = (
        workspace_root
        / "src"
        / "bot_controller"
        / "src"
        / "generated"
    )
    static_output_path = generated_dir / "wheel_odom_sigma_static_precomputes.inl"
    dynamic_output_path = generated_dir / "wheel_odom_sigma_dynamic_precomputes.inl"

    cpp_symbol_map = {
        phi_L_km1: sp.Symbol("phi_L_previous"),
        phi_L_k: sp.Symbol("phi_L_current"),
        phi_R_km1: sp.Symbol("phi_R_previous"),
        phi_R_k: sp.Symbol("phi_R_current"),
        y_c: sp.Symbol("y_c"),
        b: sp.Symbol("b"),
        r_L: sp.Symbol("r_L"),
        r_R: sp.Symbol("r_R"),
        sigma_yc: sp.Symbol("sigma_yc"),
        sigma_b: sp.Symbol("sigma_b"),
        sigma_rL: sp.Symbol("sigma_rL"),
        sigma_rR: sp.Symbol("sigma_rR"),
        sigma_phi_L: sp.Symbol("sigma_phi_L"),
        sigma_phi_R: sp.Symbol("sigma_phi_R"),
    }

    def cpp_expr(expr):
        return sp.ccode(expr.xreplace(cpp_symbol_map))

    generated_dir.mkdir(parents=True, exist_ok=True)

    static_lines = [
        "// This file is generated from notebooks/ua_diff_drive_model.ipynb,",
        "// section: Generate Factors / static precomputes.",
        "// Do not edit this file manually; update the notebook and regenerate it.",
        "//",
        "// Expected variables in the including scope:",
        "//   y_c, b, r_L, r_R,",
        "//   sigma_yc, sigma_b, sigma_rL, sigma_rR, sigma_phi_L, sigma_phi_R.",
        "",
    ]

    for symbol, expression in Sigma_rel_static_replacements:
        static_lines.append(f"const double {symbol} = {cpp_expr(expression)};")

    dynamic_lines = [
        "// This file is generated from notebooks/ua_diff_drive_model.ipynb,",
        "// section: Generate Factors / dynamic precomputes.",
        "// Do not edit this file manually; update the notebook and regenerate it.",
        "//",
        "// Expected variables in the including scope:",
        "//   phi_L_previous, phi_L_current, phi_R_previous, phi_R_current,",
        "//   y_c, b, r_L, r_R,",
        "//   sigma_yc, sigma_b, sigma_rL, sigma_rR, sigma_phi_L, sigma_phi_R,",
        "//   and q_static_* values from wheel_odom_sigma_static_precomputes.inl.",
        "",
    ]

    for symbol, expression in Sigma_rel_dynamic_replacements:
        dynamic_lines.append(f"const double {symbol} = {cpp_expr(expression)};")

    dynamic_lines.append("")
    for symbol, expression in zip(Sigma_rel_unique_names, Sigma_rel_unique_optimized):
        dynamic_lines.append(f"const double {symbol} = {cpp_expr(expression)};")

    static_output_path.write_text("\n".join(static_lines) + "\n")
    dynamic_output_path.write_text("\n".join(dynamic_lines) + "\n")
    static_output_path, dynamic_output_path
